# Bug Severity — Exploratory Data Analysis

This notebook walks through the raw Eclipse/Bugzilla bug report dataset and
answers the questions the classifier is built on:

1. How balanced are the severity classes?
2. What does the text look like (length, common tokens)?
3. Are some components dominated by particular severities?

Run `python -m src.data_loader --make-sample` first if you don't have a real
dataset yet — it will write `data/raw/sample.csv` that this notebook can load.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_bug_data
from src.preprocessor import clean_series

sns.set_theme(style='whitegrid')

In [ ]:
# Adjust path to your real dataset if available
CSV_PATH = ROOT / 'data' / 'raw' / 'sample.csv'
df = load_bug_data(CSV_PATH)
df.head()

## 1. Class distribution

In [ ]:
counts = df['severity'].value_counts()
ax = counts.plot(kind='bar', color='steelblue')
ax.set_title('Bug severity distribution')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 2. Text length

In [ ]:
df['length'] = df['text'].str.len()
sns.boxplot(data=df, x='severity', y='length')
plt.xticks(rotation=30, ha='right')
plt.title('Text length by severity')
plt.tight_layout()
plt.show()

## 3. Cleaned text preview

In [ ]:
df['clean_text'] = clean_series(df['text'].tolist())
df[['severity', 'text', 'clean_text']].sample(5, random_state=7)

## 4. Component × Severity

In [ ]:
pivot = (
    df.groupby(['component', 'severity']).size().unstack(fill_value=0)
)
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Component × Severity counts')
plt.tight_layout()
plt.show()